# Chapter 06: Native Auto-Merge vs Your Own Merge Call (Reference)

## Learning Objectives

- Distinguish 'enroll' (--auto) from 'execute now' (direct merge call)
- Call gh pr merge --auto and read its response correctly
- Attempt a direct merge call and read its response correctly
- State why GITHUB_TOKEN cannot enable auto-merge

## Setup

The next cell sets up reproducibility, the `PRA_MODE` fixture/live toggle, and inserts the repo root onto `sys.path` so `pr_automerge` is importable. You should see `PRA_MODE = 'fixture'` printed (unless you've set it to `live`).

In [ ]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

# Reproducibility -- always set before any stochastic operation
RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

# PRA_ environment variables -- fixture mode by default so this notebook runs
# identically for every reader. Set PRA_MODE=live and PRA_REPO=owner/name before
# starting Jupyter to run this against the real sandbox repo instead.
PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")
OUTPUT_DIR = Path(os.environ.get("PRA_OUTPUT_DIR", "output"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run in live mode"

print(f"PRA_MODE = {PRA_MODE!r}")
print(f"RANDOM_STATE = {RANDOM_STATE}")


## 1. Enable Native Auto-Merge

The next cell calls `enable_native_auto_merge` from `labs/lab_06_merge_modes.py`. In fixture mode this simulates the response; you should see `enrolled=True` with a rationale that says 'queued, not merged yet' -- that phrase is the whole chapter.

In [ ]:
from labs.lab_06_merge_modes import enable_native_auto_merge

result = enable_native_auto_merge("example/example", 1)
print(result)

## 2. Attempt a Direct Merge

The next cell attempts the direct, synchronous merge path instead. You should see `merged=True` with a rationale describing an immediate merge -- contrast this response shape with the previous cell's.

In [ ]:
from labs.lab_06_merge_modes import attempt_direct_merge

result = attempt_direct_merge("example/example", 1)
print(result)

## 3. The GraphQL Restriction, Live

The next cell only does something in `PRA_MODE=live`: it attempts `gh pr merge --auto` against a real PR using the default token, which Chapter 06 Section 9 predicts will fail with a GraphQL permission error. In fixture mode it just explains what would happen.

In [ ]:
if PRA_MODE == "live":
    import subprocess
    r = subprocess.run(
        ["gh", "pr", "merge", "1", "--repo", PRA_REPO, "--auto", "--squash"],
        capture_output=True, text=True,
    )
    print(r.returncode, r.stderr.strip())
else:
    print("Fixture mode: this would print a GraphQL permission error")
    print("with GITHUB_TOKEN -- see Chapter 06 Section 9.")

## Takeaways & Next Steps

This notebook's takeaways are the numbers you just produced above, not abstract claims -- re-read the printed output from each section before moving on.

In [ ]:
print("Re-run this notebook with PRA_MODE=live to see it against the real sandbox repo.")

---

📖 **Reading companion:** [Chapter 06: Native Auto-Merge vs Your Own Merge Call](../learning_modules/chapter_06_native_automerge.md)
